# Chapter 17
## Frequency-Current Curves
- Code by : [Abolfazl Ziaeemehr](https://github.com/Ziaeemehr)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ITNG/ModelingNeuralDynamics/blob/main/python/chapter17.ipynb)

## About this chapter

Firing-rate (f-I) curves distinguish the three excitability types from
earlier chapters directly: type-1 models (theta, LIF, WB near its SNIC)
have a continuous $f\to0$ onset; type-2/3 models (INaP+IK, Erisir, reduced
HH, RTM+M-current) can show hysteresis, with different onset currents
$I_c$ (upward sweep) and $I_\ast$ (downward sweep) for the same model.
Each model's frequency is measured the same way: integrate to steady
state (or up to 4 spikes) at each $I$, continuing from the previous $I$'s
final state so a forward and backward sweep can reveal hysteresis.

See [`README.md`](chapter17.md) for the full guide, including suggested
order and related chapters. Several cells below run a MATLAB-style
forward+backward sweep over 31 currents, each integrated for up to
several seconds of simulated time; the inner time-stepping loop is
JIT-compiled with numba, so after the first (one-time compile) call
each sweep takes well under a second instead of tens of seconds to
minutes.

In [ ]:
import subprocess
import sys
if "google.colab" in sys.modules:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "modelingneuraldynamics"], check=True)

In [ ]:
import math
import numpy as np
from numpy import exp
import matplotlib.pyplot as plt
from ipywidgets import interact
from numba import njit

In [ ]:
def plot_f_i_curve(f_forward, f_backward, I_c, I_star, i_ext_vec, xlim, ylim):
    print(f"I_c = {I_c}")
    print(f"I_star = {I_star}")
    plt.figure(figsize=(7, 3.5))
    plt.plot(i_ext_vec, f_forward, '.k', markersize=15, label='forward')
    plt.plot(i_ext_vec, f_backward, 'ok', markersize=10, markerfacecolor='none',
             linewidth=1, label='backward')
    plt.xlim(*xlim)
    plt.ylim(*ylim)
    plt.xlabel('$I$')
    plt.ylabel('$f$')
    plt.tight_layout()
    plt.show()


def plot_wb_f_i_curve_at_onset(f_vec, i_ext_vec, I_c, C, i_ext_low, i_ext_high):
    print(f"I_c = {I_c}")
    print(f"C = {C}")
    plt.figure(figsize=(7, 4))
    I = I_c + np.arange(1001) / 1000 * (i_ext_high - I_c)
    plt.plot(I, C * np.sqrt(I - I_c), '-r', linewidth=2)
    plt.plot([i_ext_low, I_c], [0, 0], '-r', linewidth=4)
    plt.plot(i_ext_vec, f_vec, '.k', markersize=20)
    plt.xlabel('$I$')
    plt.ylabel('$f$')
    plt.xlim(i_ext_low, i_ext_high)
    plt.ylim(0, f_vec.max() * 1.1)
    plt.xticks([0.1600, 0.1601, 0.1602])
    plt.tight_layout()
    plt.show()

## LIF F-I Curve

Closed form: $f=1000/\tau_m\ln(\tau_m I/(\tau_m I-1))$ above threshold.

In [ ]:
def simulate_lif_f_i_curve(tau_m=10.0, n=1000):
    ind = np.arange(1, n + 1)
    I = 0.1 + np.exp(-n / ind) * np.exp(1) * 0.1
    T = tau_m * np.log(tau_m * I / (tau_m * I - 1))
    f = 1000.0 / T
    return I, f


def plot_lif_f_i_curve(I, f):
    plt.figure(figsize=(7, 3.5))
    plt.plot(I, f, '-k', linewidth=2)
    plt.plot([0, 0.1], [0, 0], '-k', linewidth=4)
    plt.xlabel('$I$')
    plt.ylabel('$f$')
    plt.xlim(0, 0.2)
    plt.ylim(0, 150)
    plt.tight_layout()
    plt.show()

In [ ]:
plot_lif_f_i_curve(*simulate_lif_f_i_curve())

## Theta Neuron F-I Curve

Closed form: $f=1000\sqrt{2I-1}/\pi$ above threshold $I=1/2$.

In [ ]:
def simulate_theta_f_i_curve(n=200):
    I = 1 / 2 + np.arange(n + 1) / n * 0.1
    f = 1000 * np.sqrt(2 * I - 1) / np.pi
    return I, f


def plot_theta_f_i_curve(I, f):
    plt.figure(figsize=(7, 3.5))
    plt.plot(I, f, '-k', linewidth=2)
    plt.plot([0, 1 / 2], [0, 0], '-k', linewidth=4)
    plt.xlabel('$I$')
    plt.ylabel('$f$')
    plt.xlim(0.4, 0.6)
    plt.ylim(0, 150)
    plt.tight_layout()
    plt.show()

In [ ]:
plot_theta_f_i_curve(*simulate_theta_f_i_curve())

## $I_{Na,p}+I_K$ F-I Curve (Hysteresis)

In [ ]:
@njit
def _inapik_run_to_frequency(i_ext, v, m, n, c, g_na, g_k, g_l, v_na, v_k, v_l,
                              tau_n, dt, dt05, t_max_steps, N):
    win_maxv = win_minv = v
    win_maxm = win_minm = m
    win_maxn = win_minn = n
    num_spikes = 0
    t3 = 0.0
    t4 = 0.0

    for k in range(1, t_max_steps + 1):
        v_prev = v
        n_inf_v = 1.0 / (1 + math.exp((-25 - v) / 5))
        v_inc = (g_na * m * (v_na - v) + g_k * n * (v_k - v) + g_l * (v_l - v) + i_ext) / c
        n_inc = (n_inf_v - n) / tau_n

        v_tmp = v + dt05 * v_inc
        m_tmp = 1.0 / (1 + math.exp((-20 - v_tmp) / 15))
        n_tmp = n + dt05 * n_inc
        n_inf_vtmp = 1.0 / (1 + math.exp((-25 - v_tmp) / 5))

        v_inc = (g_na * m_tmp * (v_na - v_tmp) + g_k * n_tmp * (v_k - v_tmp) + g_l * (v_l - v_tmp) + i_ext) / c
        n_inc = (n_inf_vtmp - n_tmp) / tau_n

        v = v + dt * v_inc
        m = 1.0 / (1 + math.exp((-20 - v) / 15))
        n = n + dt * n_inc

        win_maxv = max(win_maxv, v); win_minv = min(win_minv, v)
        win_maxm = max(win_maxm, m); win_minm = min(win_minm, m)
        win_maxn = max(win_maxn, n); win_minn = min(win_minn, n)

        if v < -20 and v_prev >= -20:
            num_spikes += 1
            ts = (k * dt * (20 + v_prev) + (k - 1) * dt * (-20 - v)) / (v_prev - v)
            if num_spikes == 3:
                t3 = ts
            elif num_spikes == 4:
                t4 = ts
                return 1000.0 / (t4 - t3), v, m, n, 1

        if k % N == 0:
            if (win_maxv - win_minv) < 1e-4 * abs(win_maxv + win_minv) and \
               (win_maxm - win_minm) < 1e-4 * abs(win_maxm + win_minm) and \
               (win_maxn - win_minn) < 1e-4 * abs(win_maxn + win_minn):
                return 0.0, v, m, n, 0
            win_maxv = win_minv = v
            win_maxm = win_minm = m
            win_maxn = win_minn = n

    return 0.0, v, m, n, -1


def simulate_inapik_f_i_curve(c=1.0, g_na=20.0, g_k=10.0, g_l=8.0,
                               v_na=60.0, v_k=-90.0, v_l=-80.0, tau_n=0.15,
                               dt=0.002, t_max=50000.0, i_ext_vec=None):
    dt05 = dt / 2
    N = round(1000 / dt)
    t_max_steps = round(t_max / dt)

    def m_inf(v):
        return 1.0 / (1 + exp((-20 - v) / 15))

    def run_to_frequency(i_ext, v, m, n):
        freq, v, m, n, status = _inapik_run_to_frequency(
            i_ext, v, m, n, c, g_na, g_k, g_l, v_na, v_k, v_l, tau_n, dt, dt05, t_max_steps, N)
        if status == -1:
            raise RuntimeError(f"did not settle within {t_max_steps} steps at I={i_ext}")
        return freq, v, m, n

    if i_ext_vec is None:
        i_ext_vec = -4 + np.arange(31) / 30 * 12

    f_forward = np.zeros(len(i_ext_vec))
    v, m, n = -70.0, m_inf(-70.0), 0.6
    for ijk, i_ext in enumerate(i_ext_vec):
        f_forward[ijk], v, m, n = run_to_frequency(i_ext, v, m, n)

    f_backward = np.zeros(len(i_ext_vec))
    for ijk in range(len(i_ext_vec) - 1, -1, -1):
        f_backward[ijk], v, m, n = run_to_frequency(i_ext_vec[ijk], v, m, n)

    ind = np.where(f_forward == 0)[0].max()
    I_c = (i_ext_vec[ind] + i_ext_vec[ind + 1]) / 2
    ind = np.where(f_backward == 0)[0].max()
    I_star = (i_ext_vec[ind] + i_ext_vec[ind + 1]) / 2

    return f_forward, f_backward, I_c, I_star, i_ext_vec

In [ ]:
plot_f_i_curve(*simulate_inapik_f_i_curve(), xlim=(-3, 7), ylim=(0, 1200))

## Reduced HH F-I Curve (Hysteresis)

In [ ]:
@njit
def _hh_reduced_alpha_m(v):
    if abs(v + 45) > 1e-8:
        return (v + 45) / 10.0 / (1 - math.exp(-(v + 45) / 10))
    return 1.0


@njit
def _hh_reduced_m_inf(v):
    am = _hh_reduced_alpha_m(v)
    bm = 4 * math.exp(-(v + 70) / 18)
    return am / (am + bm)


@njit
def _hh_reduced_run_to_frequency(i_ext, v, n, c, g_na, g_k, g_l, v_na, v_k, v_l,
                                  dt, dt05, t_max_steps, N):
    m = _hh_reduced_m_inf(v)
    h = 0.83 - n
    win_maxv = win_minv = v
    win_maxm = win_minm = m
    win_maxh = win_minh = h
    win_maxn = win_minn = n
    num_spikes = 0
    t3 = 0.0
    t4 = 0.0

    for k in range(1, t_max_steps + 1):
        v_prev = v
        alpha_n_v = 0.01 * (-60.0 - v) / (math.exp((-60 - v) / 10) - 1)
        beta_n_v = 0.125 * math.exp(-(v + 70) / 80)

        v_inc = (g_na * math.pow(m, 3.0) * h * (v_na - v) + g_k * math.pow(n, 4.0) * (v_k - v)
                 + g_l * (v_l - v) + i_ext) / c
        n_inc = alpha_n_v * (1 - n) - beta_n_v * n

        v_tmp = v + dt05 * v_inc
        n_tmp = n + dt05 * n_inc
        m_tmp = _hh_reduced_m_inf(v_tmp)
        h_tmp = 0.83 - n_tmp

        alpha_n_vtmp = 0.01 * (-60.0 - v_tmp) / (math.exp((-60 - v_tmp) / 10) - 1)
        beta_n_vtmp = 0.125 * math.exp(-(v_tmp + 70) / 80)

        v_inc = (g_na * math.pow(m_tmp, 3.0) * h_tmp * (v_na - v_tmp) + g_k * math.pow(n_tmp, 4.0) * (v_k - v_tmp)
                 + g_l * (v_l - v_tmp) + i_ext) / c
        n_inc = alpha_n_vtmp * (1 - n_tmp) - beta_n_vtmp * n_tmp

        v = v + dt * v_inc
        n = n + dt * n_inc
        m = _hh_reduced_m_inf(v)
        h = 0.83 - n

        win_maxv = max(win_maxv, v); win_minv = min(win_minv, v)
        win_maxm = max(win_maxm, m); win_minm = min(win_minm, m)
        win_maxh = max(win_maxh, h); win_minh = min(win_minh, h)
        win_maxn = max(win_maxn, n); win_minn = min(win_minn, n)

        if v < -20 and v_prev >= -20:
            num_spikes += 1
            ts = (k * dt * (20 + v_prev) + (k - 1) * dt * (-20 - v)) / (v_prev - v)
            if num_spikes == 3:
                t3 = ts
            elif num_spikes == 4:
                t4 = ts
                return 1000.0 / (t4 - t3), v, n, 1

        if k % N == 0:
            if (win_maxv - win_minv) < 1e-4 * abs(win_maxv + win_minv) and \
               (win_maxm - win_minm) < 1e-4 * abs(win_maxm + win_minm) and \
               (win_maxh - win_minh) < 1e-4 * abs(win_maxh + win_minh) and \
               (win_maxn - win_minn) < 1e-4 * abs(win_maxn + win_minn):
                return 0.0, v, n, 0
            win_maxv = win_minv = v
            win_maxm = win_minm = m
            win_maxh = win_minh = h
            win_maxn = win_minn = n

    return 0.0, v, n, -1


def simulate_hh_reduced_f_i_curve(c=1.0, g_k=36.0, g_na=120.0, g_l=0.3,
                                   v_k=-82.0, v_na=45.0, v_l=-59.0,
                                   dt=0.01, t_max=5000.0, i_ext_vec=None):
    dt05 = dt / 2
    N = round(1000 / dt)
    t_max_steps = round(t_max / dt)

    def run_to_frequency(i_ext, v, n):
        freq, v, n, status = _hh_reduced_run_to_frequency(
            i_ext, v, n, c, g_na, g_k, g_l, v_na, v_k, v_l, dt, dt05, t_max_steps, N)
        if status == -1:
            raise RuntimeError(f"did not settle within {t_max_steps} steps at I={i_ext}")
        return freq, v, n

    if i_ext_vec is None:
        i_ext_vec = 3 + np.arange(31) / 30 * 10

    f_forward = np.zeros(len(i_ext_vec))
    v, n = -70.0, 0.6
    for ijk, i_ext in enumerate(i_ext_vec):
        f_forward[ijk], v, n = run_to_frequency(i_ext, v, n)

    f_backward = np.zeros(len(i_ext_vec))
    for ijk in range(len(i_ext_vec) - 1, -1, -1):
        f_backward[ijk], v, n = run_to_frequency(i_ext_vec[ijk], v, n)

    ind = np.where(f_forward == 0)[0].max()
    I_c = (i_ext_vec[ind] + i_ext_vec[ind + 1]) / 2
    ind = np.where(f_backward == 0)[0].max()
    I_star = (i_ext_vec[ind] + i_ext_vec[ind + 1]) / 2

    return f_forward, f_backward, I_c, I_star, i_ext_vec

In [ ]:
plot_f_i_curve(*simulate_hh_reduced_f_i_curve(), xlim=(3, 13), ylim=(0, 100))

## Erisir F-I Curve (Hysteresis)

In [ ]:
@njit
def _erisir_run_to_frequency(i_ext, v, m, h, n, c, g_na, g_k, g_l, v_na, v_k, v_l,
                              dt, dt05, t_max_steps, N):
    win_maxv = win_minv = v
    win_maxm = win_minm = m
    win_maxh = win_minh = h
    win_maxn = win_minn = n
    num_spikes = 0
    t3 = 0.0
    t4 = 0.0

    for k in range(1, t_max_steps + 1):
        v_prev = v
        alpha_h_v = 0.0035 / math.exp(v / 24.186)
        alpha_n_v = (95 - v) / (math.exp((95 - v) / 11.8) - 1)
        beta_h_v = -0.017 * (v + 51.25) / (math.exp(-(v + 51.25) / 5.2) - 1)
        beta_n_v = 0.025 / math.exp(v / 22.222)

        v_inc = (g_na * math.pow(m, 3.0) * h * (v_na - v) + g_k * math.pow(n, 2.0) * (v_k - v)
                 + g_l * (v_l - v) + i_ext) / c
        h_inc = alpha_h_v * (1 - h) - beta_h_v * h
        n_inc = alpha_n_v * (1 - n) - beta_n_v * n

        v_tmp = v + dt05 * v_inc
        alpha_m_vtmp = 40 * (75.5 - v_tmp) / (math.exp((75.5 - v_tmp) / 13.5) - 1)
        beta_m_vtmp = 1.2262 / math.exp(v_tmp / 42.248)
        m_tmp = alpha_m_vtmp / (alpha_m_vtmp + beta_m_vtmp)
        h_tmp = h + dt05 * h_inc
        n_tmp = n + dt05 * n_inc

        alpha_h_vtmp = 0.0035 / math.exp(v_tmp / 24.186)
        alpha_n_vtmp = (95 - v_tmp) / (math.exp((95 - v_tmp) / 11.8) - 1)
        beta_h_vtmp = -0.017 * (v_tmp + 51.25) / (math.exp(-(v_tmp + 51.25) / 5.2) - 1)
        beta_n_vtmp = 0.025 / math.exp(v_tmp / 22.222)

        v_inc = (g_na * math.pow(m_tmp, 3.0) * h_tmp * (v_na - v_tmp) + g_k * math.pow(n_tmp, 2.0) * (v_k - v_tmp)
                 + g_l * (v_l - v_tmp) + i_ext) / c
        h_inc = alpha_h_vtmp * (1 - h_tmp) - beta_h_vtmp * h_tmp
        n_inc = alpha_n_vtmp * (1 - n_tmp) - beta_n_vtmp * n_tmp

        v = v + dt * v_inc
        alpha_m_v = 40 * (75.5 - v) / (math.exp((75.5 - v) / 13.5) - 1)
        beta_m_v = 1.2262 / math.exp(v / 42.248)
        m = alpha_m_v / (alpha_m_v + beta_m_v)
        h = h + dt * h_inc
        n = n + dt * n_inc

        win_maxv = max(win_maxv, v); win_minv = min(win_minv, v)
        win_maxm = max(win_maxm, m); win_minm = min(win_minm, m)
        win_maxh = max(win_maxh, h); win_minh = min(win_minh, h)
        win_maxn = max(win_maxn, n); win_minn = min(win_minn, n)

        if v < -20 and v_prev >= -20:
            num_spikes += 1
            ts = (k * dt * (20 + v_prev) + (k - 1) * dt * (-20 - v)) / (v_prev - v)
            if num_spikes == 3:
                t3 = ts
            elif num_spikes == 4:
                t4 = ts
                return 1000.0 / (t4 - t3), v, m, h, n, 1

        if k % N == 0:
            if (win_maxv - win_minv) < 1e-4 * abs(win_maxv + win_minv) and \
               (win_maxm - win_minm) < 1e-4 * abs(win_maxm + win_minm) and \
               (win_maxh - win_minh) < 1e-4 * abs(win_maxh + win_minh) and \
               (win_maxn - win_minn) < 1e-4 * abs(win_maxn + win_minn):
                return 0.0, v, m, h, n, 0
            win_maxv = win_minv = v
            win_maxm = win_minm = m
            win_maxh = win_minh = h
            win_maxn = win_minn = n

    return 0.0, v, m, h, n, -1


def simulate_erisir_f_i_curve(c=1.0, g_k=224.0, g_na=112.0, g_l=0.5,
                               v_k=-90.0, v_na=60.0, v_l=-70.0,
                               dt=0.01, t_max=10000.0, i_ext_vec=None):
    dt05 = dt / 2
    N = round(1000 / dt)
    t_max_steps = round(t_max / dt)

    def alpha_m(v):
        return 40 * (75.5 - v) / (exp((75.5 - v) / 13.5) - 1)

    def beta_m(v):
        return 1.2262 / exp(v / 42.248)

    def m_inf(v):
        return alpha_m(v) / (alpha_m(v) + beta_m(v))

    def run_to_frequency(i_ext, v, m, h, n):
        freq, v, m, h, n, status = _erisir_run_to_frequency(
            i_ext, v, m, h, n, c, g_na, g_k, g_l, v_na, v_k, v_l, dt, dt05, t_max_steps, N)
        if status == -1:
            raise RuntimeError(f"did not settle within {t_max_steps} steps at I={i_ext}")
        return freq, v, m, h, n

    if i_ext_vec is None:
        i_ext_vec = 6 + np.arange(31) / 30 * 1.5

    f_forward = np.zeros(len(i_ext_vec))
    v, m, h, n = -70.0, m_inf(-70.0), 0.7, 0.6
    for ijk, i_ext in enumerate(i_ext_vec):
        f_forward[ijk], v, m, h, n = run_to_frequency(i_ext, v, m, h, n)

    f_backward = np.zeros(len(i_ext_vec))
    for ijk in range(len(i_ext_vec) - 1, -1, -1):
        f_backward[ijk], v, m, h, n = run_to_frequency(i_ext_vec[ijk], v, m, h, n)

    ind = np.where(f_forward == 0)[0].max()
    I_c = (i_ext_vec[ind] + i_ext_vec[ind + 1]) / 2
    ind = np.where(f_backward == 0)[0].max()
    I_star = (i_ext_vec[ind] + i_ext_vec[ind + 1]) / 2

    return f_forward, f_backward, I_c, I_star, i_ext_vec

In [ ]:
plot_f_i_curve(*simulate_erisir_f_i_curve(), xlim=(6, 7.5), ylim=(0, 80))

## Wang-Buzsaki F-I Curve (Type 1, No Hysteresis)

In [ ]:
@njit
def _wb_alpha_m(v):
    return 0.1 * (v + 35) / (1 - math.exp(-(v + 35) / 10))


@njit
def _wb_beta_m(v):
    return 4 * math.exp(-(v + 60) / 18)


@njit
def _wb_m_inf(v):
    am = _wb_alpha_m(v)
    return am / (am + _wb_beta_m(v))


@njit
def _wb_run_to_frequency(i_ext, v, m, h, n, c, g_na, g_k, g_l, v_na, v_k, v_l,
                          dt, dt05, t_max_steps, N):
    win_maxv = win_minv = v
    win_maxm = win_minm = m
    win_maxh = win_minh = h
    win_maxn = win_minn = n
    num_spikes = 0
    t3 = 0.0
    t4 = 0.0

    for k in range(1, t_max_steps + 1):
        v_prev = v
        alpha_h_v = 0.35 * math.exp(-(v + 58) / 20)
        alpha_n_v = 0.05 * (v + 34) / (1 - math.exp(-0.1 * (v + 34)))
        beta_h_v = 5.0 / (math.exp(-0.1 * (v + 28)) + 1)
        beta_n_v = 0.625 * math.exp(-(v + 44) / 80)

        v_inc = (g_na * math.pow(m, 3.0) * h * (v_na - v) + g_k * math.pow(n, 4.0) * (v_k - v)
                 + g_l * (v_l - v) + i_ext) / c
        h_inc = alpha_h_v * (1 - h) - beta_h_v * h
        n_inc = alpha_n_v * (1 - n) - beta_n_v * n

        v_tmp = v + dt05 * v_inc
        m_tmp = _wb_m_inf(v_tmp)
        h_tmp = h + dt05 * h_inc
        n_tmp = n + dt05 * n_inc

        alpha_h_vtmp = 0.35 * math.exp(-(v_tmp + 58) / 20)
        alpha_n_vtmp = 0.05 * (v_tmp + 34) / (1 - math.exp(-0.1 * (v_tmp + 34)))
        beta_h_vtmp = 5.0 / (math.exp(-0.1 * (v_tmp + 28)) + 1)
        beta_n_vtmp = 0.625 * math.exp(-(v_tmp + 44) / 80)

        v_inc = (g_na * math.pow(m_tmp, 3.0) * h_tmp * (v_na - v_tmp) + g_k * math.pow(n_tmp, 4.0) * (v_k - v_tmp)
                 + g_l * (v_l - v_tmp) + i_ext) / c
        h_inc = alpha_h_vtmp * (1 - h_tmp) - beta_h_vtmp * h_tmp
        n_inc = alpha_n_vtmp * (1 - n_tmp) - beta_n_vtmp * n_tmp

        v = v + dt * v_inc
        m = _wb_m_inf(v)
        h = h + dt * h_inc
        n = n + dt * n_inc

        win_maxv = max(win_maxv, v); win_minv = min(win_minv, v)
        win_maxm = max(win_maxm, m); win_minm = min(win_minm, m)
        win_maxh = max(win_maxh, h); win_minh = min(win_minh, h)
        win_maxn = max(win_maxn, n); win_minn = min(win_minn, n)

        if v < -20 and v_prev >= -20:
            num_spikes += 1
            ts = (k * dt * (20 + v_prev) + (k - 1) * dt * (-20 - v)) / (v_prev - v)
            if num_spikes == 3:
                t3 = ts
            elif num_spikes == 4:
                t4 = ts
                return 1000.0 / (t4 - t3), v, m, h, n, 1

        if k % N == 0:
            if (win_maxv - win_minv) < 1e-4 * abs(win_maxv + win_minv) and \
               (win_maxm - win_minm) < 1e-4 * abs(win_maxm + win_minm) and \
               (win_maxh - win_minh) < 1e-4 * abs(win_maxh + win_minh) and \
               (win_maxn - win_minn) < 1e-4 * abs(win_maxn + win_minn):
                return 0.0, v, m, h, n, 0
            win_maxv = win_minv = v
            win_maxm = win_minm = m
            win_maxh = win_minh = h
            win_maxn = win_minn = n

    return 0.0, v, m, h, n, -1

In [ ]:
def simulate_wb_f_i_curve(c=1.0, g_k=9.0, g_na=35.0, g_l=0.1,
                           v_k=-90.0, v_na=55.0, v_l=-65.0,
                           dt=0.01, t_max=5000.0, i_ext_vec=None):
    dt05 = dt / 2
    N = round(1000 / dt)
    t_max_steps = round(t_max / dt)

    def run_to_frequency(i_ext, v, m, h, n):
        freq, v, m, h, n, status = _wb_run_to_frequency(
            i_ext, v, m, h, n, c, g_na, g_k, g_l, v_na, v_k, v_l, dt, dt05, t_max_steps, N)
        if status == -1:
            raise RuntimeError(f"did not settle within {t_max_steps} steps at I={i_ext}")
        return freq, v, m, h, n

    if i_ext_vec is None:
        i_ext_low, i_ext_high = 0.0, 1.0
        i_ext_vec = i_ext_low + np.arange(31) / 30 * (i_ext_high - i_ext_low)

    f_forward = np.zeros(len(i_ext_vec))
    v, m, h, n = -70.0, _wb_m_inf(-70.0), 0.7, 0.6
    for ijk, i_ext in enumerate(i_ext_vec):
        f_forward[ijk], v, m, h, n = run_to_frequency(i_ext, v, m, h, n)

    f_backward = np.zeros(len(i_ext_vec))
    for ijk in range(len(i_ext_vec) - 1, -1, -1):
        f_backward[ijk], v, m, h, n = run_to_frequency(i_ext_vec[ijk], v, m, h, n)

    ind = np.where(f_forward == 0)[0].max()
    I_c = (i_ext_vec[ind] + i_ext_vec[ind + 1]) / 2
    ind = np.where(f_backward == 0)[0].max()
    I_star = (i_ext_vec[ind] + i_ext_vec[ind + 1]) / 2

    return f_forward, f_backward, I_c, I_star, i_ext_vec

In [ ]:
f_forward, f_backward, I_c, I_star, i_ext_vec = simulate_wb_f_i_curve()
plot_f_i_curve(f_forward, f_backward, I_c, I_star, i_ext_vec,
               xlim=(0, 1), ylim=(0, max(f_forward.max(), f_backward.max()) * 1.1))

## Wang-Buzsaki F-I Curve, Zoomed at Onset

No hysteresis for WB, so $f\to0$ continuously; this zooms into an
exponentially narrow window of $I$ near threshold and fits
$f=C\sqrt{I-I_c}$.

In [ ]:
def simulate_wb_f_i_curve_at_onset(c=1.0, g_k=9.0, g_na=35.0, g_l=0.1,
                                    v_k=-90.0, v_na=55.0, v_l=-65.0,
                                    dt=0.01, t_max=200000.0,
                                    i_ext_low=0.1600, i_ext_high=0.1602):
    dt05 = dt / 2
    N = round(1000 / dt)
    t_max_steps = round(t_max / dt)

    def run_to_frequency(i_ext, v, m, h, n):
        freq, v, m, h, n, status = _wb_run_to_frequency(
            i_ext, v, m, h, n, c, g_na, g_k, g_l, v_na, v_k, v_l, dt, dt05, t_max_steps, N)
        if status == -1:
            raise RuntimeError(f"did not settle within {t_max_steps} steps at I={i_ext}")
        return freq, v, m, h, n

    i_ext_vec = i_ext_low + np.arange(11) / 10 * (i_ext_high - i_ext_low)
    f_vec = np.zeros(len(i_ext_vec))
    v, m, h, n = -70.0, _wb_m_inf(-70.0), 0.7, 0.6
    for ijk, i_ext in enumerate(i_ext_vec):
        f_vec[ijk], v, m, h, n = run_to_frequency(i_ext, v, m, h, n)

    ind = np.where(f_vec > 0)[0]
    i_ext_vec_0 = i_ext_vec[ind]
    f_vec_0 = f_vec[ind]
    I_c_low = i_ext_vec[ind.min() - 1]
    I_c_high = i_ext_vec[ind.min()]

    alpha_vec = np.arange(101) / 100
    C_vec = np.zeros(len(alpha_vec))
    err_vec = np.zeros(len(alpha_vec))
    for ijk, alpha in enumerate(alpha_vec):
        I_c_trial = I_c_low * alpha + I_c_high * (1 - alpha)
        with np.errstate(divide='ignore', invalid='ignore'):
            y = f_vec_0 / np.sqrt(i_ext_vec_0 - I_c_trial)
            C_vec[ijk] = y.mean()
            err_vec[ijk] = (y.max() - y.min()) / y.mean()

    ind_best = np.where(err_vec == np.nanmin(err_vec))[0].min()
    alpha = alpha_vec[ind_best]
    I_c = I_c_low * alpha + I_c_high * (1 - alpha)
    C = C_vec[ind_best]

    return f_vec, i_ext_vec, I_c, C, i_ext_low, i_ext_high

In [ ]:
# JIT-compiled via _wb_run_to_frequency; the first call compiles (~1s),
# subsequent calls are fast even this close to the SNIC onset.
plot_wb_f_i_curve_at_onset(*simulate_wb_f_i_curve_at_onset())

## RTM with M-Current F-I Curve (Hysteresis)

In [ ]:
@njit
def _rtm_m_m_inf(v):
    am = 0.32 * (v + 54) / (1 - math.exp(-(v + 54) / 4))
    bm = 0.28 * (v + 27) / (math.exp((v + 27) / 5) - 1)
    return am / (am + bm)


@njit
def _rtm_m_run_to_frequency(i_ext, v, m, h, n, w, c, g_na, g_k, g_l, v_na, v_k, v_l,
                             g_m, dt, dt05, t_max_steps, N):
    win_maxv = win_minv = v
    win_maxm = win_minm = m
    win_maxh = win_minh = h
    win_maxn = win_minn = n
    win_maxw = win_minw = w
    num_spikes = 0
    t3 = 0.0
    t4 = 0.0

    for k in range(1, t_max_steps + 1):
        v_prev = v
        alpha_h_v = 0.128 * math.exp(-(v + 50) / 18)
        alpha_n_v = 0.032 * (v + 52) / (1 - math.exp(-(v + 52) / 5))
        beta_h_v = 4.0 / (1 + math.exp(-(v + 27) / 5))
        beta_n_v = 0.5 * math.exp(-(v + 57) / 40)
        w_inf_v = 1.0 / (1 + math.exp(-(v + 35) / 10))
        tau_w_v = 400.0 / (3.3 * math.exp((v + 35) / 20) + math.exp(-(v + 35) / 20))

        v_inc = (g_na * math.pow(m, 3.0) * h * (v_na - v) + g_k * math.pow(n, 4.0) * (v_k - v)
                 + g_l * (v_l - v) + g_m * w * (v_k - v) + i_ext) / c
        h_inc = alpha_h_v * (1 - h) - beta_h_v * h
        n_inc = alpha_n_v * (1 - n) - beta_n_v * n
        w_inc = (w_inf_v - w) / tau_w_v

        v_tmp = v + dt05 * v_inc
        m_tmp = _rtm_m_m_inf(v_tmp)
        h_tmp = h + dt05 * h_inc
        n_tmp = n + dt05 * n_inc
        w_tmp = w + dt05 * w_inc

        alpha_h_vtmp = 0.128 * math.exp(-(v_tmp + 50) / 18)
        alpha_n_vtmp = 0.032 * (v_tmp + 52) / (1 - math.exp(-(v_tmp + 52) / 5))
        beta_h_vtmp = 4.0 / (1 + math.exp(-(v_tmp + 27) / 5))
        beta_n_vtmp = 0.5 * math.exp(-(v_tmp + 57) / 40)
        w_inf_vtmp = 1.0 / (1 + math.exp(-(v_tmp + 35) / 10))
        tau_w_vtmp = 400.0 / (3.3 * math.exp((v_tmp + 35) / 20) + math.exp(-(v_tmp + 35) / 20))

        v_inc = (g_na * math.pow(m_tmp, 3.0) * h_tmp * (v_na - v_tmp) + g_k * math.pow(n_tmp, 4.0) * (v_k - v_tmp)
                 + g_l * (v_l - v_tmp) + g_m * w_tmp * (v_k - v_tmp) + i_ext) / c
        h_inc = alpha_h_vtmp * (1 - h_tmp) - beta_h_vtmp * h_tmp
        n_inc = alpha_n_vtmp * (1 - n_tmp) - beta_n_vtmp * n_tmp
        # faithful port: divides by w (not w_tmp), matching the book's
        # second-stage w_inc -- kept as-is, see notebook note below
        w_inc = (w_inf_vtmp - w) / tau_w_vtmp

        v = v + dt * v_inc
        m = _rtm_m_m_inf(v)
        h = h + dt * h_inc
        n = n + dt * n_inc
        w = w + dt * w_inc

        win_maxv = max(win_maxv, v); win_minv = min(win_minv, v)
        win_maxm = max(win_maxm, m); win_minm = min(win_minm, m)
        win_maxh = max(win_maxh, h); win_minh = min(win_minh, h)
        win_maxn = max(win_maxn, n); win_minn = min(win_minn, n)
        win_maxw = max(win_maxw, w); win_minw = min(win_minw, w)

        if v < -20 and v_prev >= -20:
            num_spikes += 1
            ts = (k * dt * (20 + v_prev) + (k - 1) * dt * (-20 - v)) / (v_prev - v)
            if num_spikes == 3:
                t3 = ts
            elif num_spikes == 4:
                t4 = ts
                return 1000.0 / (t4 - t3), v, m, h, n, w, 1

        if k % N == 0:
            if (win_maxv - win_minv) < 1e-4 * abs(win_maxv + win_minv) and \
               (win_maxm - win_minm) < 1e-4 * abs(win_maxm + win_minm) and \
               (win_maxh - win_minh) < 1e-4 * abs(win_maxh + win_minh) and \
               (win_maxn - win_minn) < 1e-4 * abs(win_maxn + win_minn) and \
               (win_maxw - win_minw) < 1e-4 * abs(win_maxw + win_minw):
                return 0.0, v, m, h, n, w, 0
            win_maxv = win_minv = v
            win_maxm = win_minm = m
            win_maxh = win_minh = h
            win_maxn = win_minn = n
            win_maxw = win_minw = w

    return 0.0, v, m, h, n, w, -1


def simulate_rtm_with_m_current_f_i(c=1.0, g_k=80.0, g_na=100.0, g_l=0.1,
                                     v_k=-100.0, v_na=50.0, v_l=-67.0, g_m=0.2,
                                     dt=0.01, t_max=100000.0, i_ext_vec=None):
    dt05 = dt / 2
    N = round(1000 / dt)
    t_max_steps = round(t_max / dt)

    def run_to_frequency(i_ext, v, m, h, n, w):
        freq, v, m, h, n, w, status = _rtm_m_run_to_frequency(
            i_ext, v, m, h, n, w, c, g_na, g_k, g_l, v_na, v_k, v_l, g_m,
            dt, dt05, t_max_steps, N)
        if status == -1:
            raise RuntimeError(f"did not settle within {t_max_steps} steps at I={i_ext}")
        return freq, v, m, h, n, w

    if i_ext_vec is None:
        i_ext_low, i_ext_high = 0.50, 0.51
        i_ext_vec = i_ext_low + np.arange(31) / 30 * (i_ext_high - i_ext_low)

    f_forward = np.zeros(len(i_ext_vec))
    v, m, h, n, w = -70.0, _rtm_m_m_inf(-70.0), 0.7, 0.6, 0.0
    for ijk, i_ext in enumerate(i_ext_vec):
        f_forward[ijk], v, m, h, n, w = run_to_frequency(i_ext, v, m, h, n, w)

    f_backward = np.zeros(len(i_ext_vec))
    for ijk in range(len(i_ext_vec) - 1, -1, -1):
        f_backward[ijk], v, m, h, n, w = run_to_frequency(i_ext_vec[ijk], v, m, h, n, w)

    ind = np.where(f_forward == 0)[0].max()
    I_c = (i_ext_vec[ind] + i_ext_vec[ind + 1]) / 2
    ind = np.where(f_backward == 0)[0].max()
    I_star = (i_ext_vec[ind] + i_ext_vec[ind + 1]) / 2

    return f_forward, f_backward, I_c, I_star, i_ext_vec

In [ ]:
f_forward, f_backward, I_c, I_star, i_ext_vec = simulate_rtm_with_m_current_f_i()
plot_f_i_curve(f_forward, f_backward, I_c, I_star, i_ext_vec,
               xlim=(0.50, 0.51), ylim=(0, max(f_forward.max(), f_backward.max()) * 1.1))

## $I_{Na,p}+I_K$ Saddle-Cycle Distance

Distance from the settled limit cycle to the saddle fixed point, as $I$
approaches the SNIC from below -- the trajectory lingers closer to the
saddle as the homoclinic-like slow passage develops.

In [ ]:
def simulate_inapik_saddle_cycle_distance(c=1.0, g_na=20.0, g_k=10.0, g_l=8.0,
                                           v_na=60.0, v_k=-90.0, v_l=-80.0, tau_n=0.15,
                                           t_final=20.0, dt=0.001, i_ext_vec=None):
    def m_inf(v):
        return 1.0 / (1 + exp((-20 - v) / 15))

    def m_inf_p(v):
        return -1.0 / (1 + exp((-20 - v) / 15)) ** 2 * exp((-20 - v) / 15) * (-1 / 15)

    def n_inf(v):
        return 1.0 / (1 + exp((-25 - v) / 5))

    def n_inf_p(v):
        return -1.0 / (1 + exp((-25 - v) / 5)) ** 2 * exp((-25 - v) / 5) * (-1 / 5)

    def f(v):
        """zeros of this function (offset by I) are the fixed points"""
        return g_na * m_inf(v) * (v_na - v) + g_k * n_inf(v) * (v_k - v) + g_l * (v_l - v)

    w_grid = -100 + np.arange(10001) / 10000 * 150
    f_grid = f(w_grid)

    def find_saddle(I):
        fi = f_grid + I
        v_star = n_star = None
        for j in np.where(fi[:-1] * fi[1:] <= 0)[0]:
            w_low, w_high = w_grid[j], w_grid[j + 1]
            while w_high - w_low > 1e-12:
                w_c = (w_low + w_high) / 2
                if (f(w_c) + I) * (f(w_high) + I) <= 0:
                    w_low = w_c
                else:
                    w_high = w_c
            v_c = (w_low + w_high) / 2
            n_c = n_inf(v_c)
            j00 = g_na * m_inf_p(v_c) * (v_na - v_c) - g_na * m_inf(v_c) - g_k * n_c - g_l
            j01 = g_k * (v_k - v_c)
            j10 = n_inf_p(v_c) / tau_n
            j11 = -1 / tau_n
            e = np.linalg.eigvals(np.array([[j00, j01], [j10, j11]]))
            if abs(e[0].imag) < 1e-12 and e[0].real * e[1].real < 0:
                v_star, n_star = v_c, n_c
        return v_star, n_star

    dt05 = dt / 2
    m_steps = round(t_final / dt)

    if i_ext_vec is None:
        low, high = -1.38, 0.0
        i_ext_vec = low + np.arange(101) / 100 * (high - low)

    d_vec = np.zeros(len(i_ext_vec))
    for ijk, i_ext in enumerate(i_ext_vec):
        v = np.zeros(m_steps + 1)
        n = np.zeros(m_steps + 1)
        v[0], n[0] = -30.0, 0.2

        for k in range(m_steps):
            m = m_inf(v[k])
            v_inc = (g_na * m * (v_na - v[k]) + g_k * n[k] * (v_k - v[k]) + g_l * (v_l - v[k]) + i_ext) / c
            n_inc = (n_inf(v[k]) - n[k]) / tau_n

            v_tmp = v[k] + dt05 * v_inc
            m_tmp = m_inf(v_tmp)
            n_tmp = n[k] + dt05 * n_inc

            v_inc = (g_na * m_tmp * (v_na - v_tmp) + g_k * n_tmp * (v_k - v_tmp) + g_l * (v_l - v_tmp) + i_ext) / c
            n_inc = (n_inf(v_tmp) - n_tmp) / tau_n

            v[k + 1] = v[k] + dt * v_inc
            n[k + 1] = n[k] + dt * n_inc

        v_star, n_star = find_saddle(i_ext)
        d_vec[ijk] = np.sqrt(((v - v_star) ** 2 + (n - n_star) ** 2).min())

    return d_vec, i_ext_vec


def plot_inapik_saddle_cycle_distance(d_vec, i_ext_vec):
    plt.figure(figsize=(7, 3.5))
    plt.plot(i_ext_vec, d_vec, '-k', linewidth=2)
    plt.xlabel('$I$')
    plt.ylabel('$d$')
    plt.tight_layout()
    plt.show()

In [ ]:
plot_inapik_saddle_cycle_distance(*simulate_inapik_saddle_cycle_distance())

## Self-Exciting Theta Neuron F-I Curve

Frequency from two different initial phases ($\theta_0=0$ and
$\theta_0=9\pi/10$), showing hysteresis from the spike-triggered
adaptation.

In [ ]:
def simulate_setn_f_i(tau_z=100.0, z_max=0.05, i_l=-0.06, i_r=0.02,
                       dt=0.01, max_num_spikes=3, t_max=1000.0, i_ext_vec=None):
    dt05 = dt / 2

    def run(i_ext, theta0):
        theta, z = theta0, 0.0
        num_spikes = 0
        t_spikes = []
        k = 0
        t = 0.0
        while num_spikes < max_num_spikes and t < t_max:
            theta_inc = 1 - np.cos(theta) + (i_ext + z) * (1 + np.cos(theta))
            z_inc = -z / tau_z + 10 * np.exp(-5 * (1 + np.cos(theta))) * (z_max - z)
            theta_tmp = theta + dt05 * theta_inc
            z_tmp = z + dt05 * z_inc
            theta_inc = 1 - np.cos(theta_tmp) + (i_ext + z_tmp) * (1 + np.cos(theta_tmp))
            z_inc = -z_tmp / tau_z + 10 * np.exp(-5 * (1 + np.cos(theta_tmp))) * (z_max - z_tmp)
            theta_next = theta + dt * theta_inc
            z = z + dt * z_inc
            k += 1
            if theta_next > np.pi:
                num_spikes += 1
                t_spike = ((k - 1) * dt * (theta_next - np.pi) + k * dt * (np.pi - theta)) / (theta_next - theta)
                t_spikes.append(t_spike)
                theta_next -= 2 * np.pi
            theta = theta_next
            t += dt

        if num_spikes == max_num_spikes:
            return 1000 / (t_spikes[-1] - t_spikes[-2])
        return 0.0

    if i_ext_vec is None:
        i_ext_vec = i_l + np.arange(31) / 30 * (i_r - i_l)

    f_low = np.array([run(i_ext, 0.0) for i_ext in i_ext_vec])
    f_high = np.array([run(i_ext, 9 / 10 * np.pi) for i_ext in i_ext_vec])
    return f_low, f_high, i_ext_vec


def plot_setn_f_i(f_low, f_high, i_ext_vec):
    plt.figure(figsize=(7, 3.5))
    plt.plot(i_ext_vec, f_low, '.k', markersize=15)
    plt.plot(i_ext_vec, f_high, 'ok', markersize=10, markerfacecolor='none', linewidth=1)
    plt.xlim(i_ext_vec.min(), i_ext_vec.max())
    plt.ylim(0, 100)
    plt.xlabel('$I$')
    plt.ylabel('$f$')
    plt.xticks(np.arange(-0.06, 0.021, 0.02))
    plt.tight_layout()
    plt.show()

In [ ]:
plot_setn_f_i(*simulate_setn_f_i())

## Legacy Full-Model F-I Curves

Three older, untested scripts kept for content parity with the original
`python/17_Frequency_Current_Curves/` folder: full (non-reduced) HH and
RTM models, integrated with classic RK4 rather than the Heun/RK2 stepper
used above. These never had dedicated test coverage even before this
notebook existed, so treat them as reference material rather than
verified results. Default parameters reproduce the original scripts;
smoke tests below use smaller scans to stay fast.

In [ ]:
def _rk4_step(x, dt, f, i_ext):
    k1 = dt * f(x, i_ext)
    k2 = dt * f(x + 0.5 * k1, i_ext)
    k3 = dt * f(x + 0.5 * k2, i_ext)
    k4 = dt * f(x + k3, i_ext)
    return x + (k1 + 2.0 * k2 + 2.0 * k3 + k4) / 6.0


def _legacy_scan(derivative, i_ext_vec, x0, t_final, dt, v_threshold=-20.0):
    """Shared forward+backward RK4 f-I scan used by the three legacy
    full-model examples below."""
    N = int(1000 / dt)
    num_steps = int(t_final / dt)

    def run_direction(i_ext_vec):
        frequencies = np.zeros(len(i_ext_vec))
        state = list(x0)
        for ii, i_ext in enumerate(i_ext_vec):
            num_spikes = 0
            t_spikes = []
            v = np.zeros(num_steps)
            m = np.zeros_like(v)
            n = np.zeros_like(v)
            h = np.zeros_like(v)
            for i in range(num_steps):
                v[i], m[i], n[i], h[i] = _rk4_step(state, dt, derivative, i_ext)
                state = [v[i], m[i], n[i], h[i]]
                if (i % N) == 0 and i > 0:
                    maxv, minv = max(v[i - N:i]), min(v[i - N:i])
                    maxm, minm = max(m[i - N:i]), min(m[i - N:i])
                    maxn, minn = max(n[i - N:i]), min(n[i - N:i])
                    maxh, minh = max(h[i - N:i]), min(h[i - N:i])
                    if ((maxv - minv) < 1e-4 * abs(maxv + minv)
                            and (maxm - minm) < 1e-4 * abs(maxm + minm)
                            and (maxh - minh) < 1e-4 * abs(maxh + minh)
                            and (maxn - minn) < 1e-4 * abs(maxn + minn)):
                        frequencies[ii] = 0.0
                        break
                if v[i - 1] < v_threshold <= v[i]:
                    num_spikes += 1
                    tmp = ((i - 1) * dt * (v[i - 1] - v_threshold)
                           + i * dt * (v_threshold - v[i])) / (v[i - 1] - v[i])
                    t_spikes.append(tmp)
                if num_spikes == 4:
                    frequencies[ii] = 1000.0 / (t_spikes[-1] - t_spikes[-2])
                    break
        return frequencies

    f_forward = run_direction(i_ext_vec)
    f_backward = run_direction(i_ext_vec[::-1])[::-1]
    return f_forward, f_backward

### Legacy Full HH F-I Curve

In [ ]:
def simulate_hh_f_i_curve_legacy(c=1.0, g_k=36.0, g_na=120.0, g_l=0.3,
                                  v_k=-82.0, v_na=45.0, v_l=-59.0,
                                  i_ext_vec=None, t_final=3000.0, dt=0.05):
    def alpha_h(v):
        return 0.07 * exp(-(v + 70.0) / 20.0)

    def alpha_m(v):
        return (v + 45.0) / 10.0 / (1 - exp(-(v + 45.0) / 10.0))

    def alpha_n(v):
        return 0.01 * (-60.0 - v) / (exp((-60.0 - v) / 10.0) - 1.0)

    def beta_h(v):
        return 1.0 / (exp(-(v + 40.0) / 10.0) + 1.0)

    def beta_m(v):
        return 4.0 * exp(-(v + 70.0) / 18.0)

    def beta_n(v):
        return 0.125 * exp(-(v + 70.0) / 80.0)

    def derivative(x0, i_ext):
        v, m, n, h = x0
        i_na = -g_na * h * m ** 3 * (v - v_na)
        i_k = -g_k * n ** 4 * (v - v_k)
        i_l = -g_l * (v - v_l)
        dv = (i_ext + i_na + i_k + i_l) / c
        dm = alpha_m(v) * (1.0 - m) - beta_m(v) * m
        dn = alpha_n(v) * (1.0 - n) - beta_n(v) * n
        dh = alpha_h(v) * (1.0 - h) - beta_h(v) * h
        return np.array([dv, dm, dn, dh])

    if i_ext_vec is None:
        i_ext_vec = np.linspace(3, 13, 23)
    x0 = [-70.0, 0.0, 0.6, 0.7]  # v, m, n, h -- m recomputed on the first RK4 step anyway
    f_forward, f_backward = _legacy_scan(derivative, i_ext_vec, x0, t_final, dt)
    return f_forward, f_backward, i_ext_vec

### Legacy Full RTM F-I Curve

In [ ]:
def simulate_rtm_f_i_curve_legacy(c=1.0, g_k=80.0, g_na=100.0, g_l=0.1,
                                   v_k=-100.0, v_na=50.0, v_l=-67.0,
                                   i_ext_vec=None, t_final=5000.0, dt=0.05):
    def alpha_h(v):
        return 0.128 * exp(-(v + 50.0) / 18.0)

    def alpha_m(v):
        return 0.32 * (v + 54) / (1.0 - exp(-(v + 54.0) / 4.0))

    def alpha_n(v):
        return 0.032 * (v + 52) / (1.0 - exp(-(v + 52.0) / 5.0))

    def beta_h(v):
        return 4.0 / (1.0 + exp(-(v + 27.0) / 5.0))

    def beta_m(v):
        return 0.28 * (v + 27.0) / (exp((v + 27.0) / 5.0) - 1.0)

    def beta_n(v):
        return 0.5 * exp(-(v + 57.0) / 40.0)

    def derivative(x0, i_ext):
        v, m, n, h = x0
        dv = (i_ext - g_na * h * m ** 3 * (v - v_na)
              - g_k * n ** 4 * (v - v_k) - g_l * (v - v_l))
        dm = alpha_m(v) * (1.0 - m) - beta_m(v) * m
        dn = alpha_n(v) * (1.0 - n) - beta_n(v) * n
        dh = alpha_h(v) * (1.0 - h) - beta_h(v) * h
        return np.array([dv, dm, dn, dh])

    if i_ext_vec is None:
        i_ext_vec = np.linspace(0, 1, 61)
    x0 = [-70.0, 0.0, 0.6, 0.7]
    f_forward, f_backward = _legacy_scan(derivative, i_ext_vec, x0, t_final, dt)
    return f_forward, f_backward, i_ext_vec

### Legacy Full RTM F-I Curve, Zoomed at Onset

Fits $f=C\sqrt{I-I_c}$ near threshold, same idea as the WB onset example
above. Note: the original script's `alpha_vec` search grid was built with
`np.arange(0, 1, 101)` (step size 101, not `np.linspace`), a bug that
collapses the search to a single candidate `alpha=0`; kept as-is here for
a faithful port of behavior actually observed in `fig_17_5.png`.

In [ ]:
def simulate_rtm_f_i_curve_at_onset_legacy(c=1.0, g_k=80.0, g_na=100.0, g_l=0.1,
                                            v_k=-100.0, v_na=50.0, v_l=-67.0,
                                            i_ext_low=0.1193, i_ext_high=0.1194,
                                            n_points=11, t_final=20000.0, dt=0.05):
    def alpha_h(v):
        return 0.128 * exp(-(v + 50.0) / 18.0)

    def alpha_m(v):
        return 0.32 * (v + 54) / (1.0 - exp(-(v + 54.0) / 4.0))

    def alpha_n(v):
        return 0.032 * (v + 52) / (1.0 - exp(-(v + 52.0) / 5.0))

    def beta_h(v):
        return 4.0 / (1.0 + exp(-(v + 27.0) / 5.0))

    def beta_m(v):
        return 0.28 * (v + 27.0) / (exp((v + 27.0) / 5.0) - 1.0)

    def beta_n(v):
        return 0.5 * exp(-(v + 57.0) / 40.0)

    def derivative(x0, i_ext):
        v, m, n, h = x0
        dv = (i_ext - g_na * h * m ** 3 * (v - v_na)
              - g_k * n ** 4 * (v - v_k) - g_l * (v - v_l))
        dm = alpha_m(v) * (1.0 - m) - beta_m(v) * m
        dn = alpha_n(v) * (1.0 - n) - beta_n(v) * n
        dh = alpha_h(v) * (1.0 - h) - beta_h(v) * h
        return np.array([dv, dm, dn, dh])

    i_ext_vec = np.linspace(i_ext_low, i_ext_high, n_points)
    x0 = [-70.0, 0.0, 0.6, 0.7]
    f_forward, _ = _legacy_scan(derivative, i_ext_vec, x0, t_final, dt)

    index = np.where(f_forward == 0)[0]
    if len(index) == 0 or index.max() + 1 >= len(i_ext_vec):
        return f_forward, i_ext_vec, None, None

    index = index.max()
    I0, f0 = i_ext_vec[index], f_forward[index]
    I_c_low, I_c_high = i_ext_vec[index], i_ext_vec[index + 1]

    # bug-for-bug port: np.arange(0, 1, 101) yields only [0.0], not a 101-point grid
    alpha_vec = np.arange(0, 1, 101)
    C_vec = np.zeros(len(alpha_vec))
    err_vec = np.zeros(len(alpha_vec))
    for ijk, alpha in enumerate(alpha_vec):
        I_c_trial = I_c_low * alpha + I_c_high * (1 - alpha)
        with np.errstate(divide='ignore', invalid='ignore'):
            y = f0 / np.sqrt(I0 - I_c_trial)
        err_vec[ijk] = (np.max(y) - np.min(y)) / np.mean(y)
        C_vec[ijk] = np.mean(y)

    ind = np.argmin(err_vec)
    I_c = I_c_low * alpha_vec[ind] + I_c_high * (1 - alpha_vec[ind])
    C = C_vec[ind]
    return f_forward, i_ext_vec, I_c, C